# 01. Extracción de Datos Geoespaciales (MapBiomas + ERA5-Land)

Objetivo de este notebook:
Conectarnos a la API de Google Earth Engine (GEE) para extraer una muestra aleatoria de datos de cobertura de suelo y variables climáticas.

Justificación del año de estudio:
Para este prototipo, extraeremos datos del año 2020. Durante este período, la región del Gran Chaco experimentó una sequía histórica severa. Este estrés hídrico extremo es el principal generador de "ruido óptico" en las imágenes satelitales, provocando que los algoritmos de clasificación generen transiciones espurias (ej. clasificar un bosque estresado como suelo desnudo).

In [1]:
import ee
import geemap
import pandas as pd
import os

# Autenticación e Inicialización en Google Earth Engine
try:
    ee.Initialize()
    print("Conexión con Google Earth Engine exitosa.")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("Autenticación y conexión exitosas.")


Successfully saved authorization token.
Autenticación y conexión exitosas.


## Carga de activos de la nube

En lugar de descargar imágenes satelitales pesadas, llamaremos a los catálogos públicos alojados en GEE:

1. Cobertura de suelo: Colección 2 de MapBiomas Argentina (Región Chaco).
2. Clima: Catálogo ERA5-Land Monthly Aggregates (ECMWF) para obtener la temperatura y precipitaciones.

In [2]:
# 1. Definir el área de estudio (Polígono de prueba en el Gran Chaco)
# Estas coordenadas abarcan una zona caliente de deforestación
aoi = ee.Geometry.Rectangle([-63.0, -27.5, -61.0, -26.0])

# 2. Cargar el Asset de la Colección 2 de MapBiomas Chaco
mapbiomas = ee.Image('projects/mapbiomas-argentina/assets/LAND-COVER/COLLECTION-2/GENERAL/CLASSIFICATION/FINAL_CLASSIFICATION/CHACO/CHACO-FINAL-v1')

# 3. Cargar el Catálogo Climático ERA5-Land (Mensual)
clima_era5 = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")

## Procesamiento espacial y generación de muestra

Se filtrarán los datos para el año 2020. Unificaremos las capas de clasificación de vegetación con los promedios climáticos anuales. Luego, generaremos 1.000 puntos aleatorios dentro de nuestra Área de Interés (AOI) para extraer el valor exacto de los píxeles.

In [3]:
anio = 2020

# Seleccionar la banda de clasificación de MapBiomas para el año 2020
banda_cobertura = f'classification_{anio}'
mapa_2020 = mapbiomas.select(banda_cobertura)

# Filtrar el clima para el año 2020 y calcular el promedio anual de las variables
clima_2020 = clima_era5.filterDate(f'{anio}-01-01', f'{anio}-12-31').mean()

# Seleccionar Temperatura (viene en Kelvin) y Precipitación
clima_seleccionado = clima_2020.select(['temperature_2m', 'total_precipitation_sum'])

# Unir la capa de vegetación con las capas de clima como si fueran "hojas de un mismo mapa"
imagen_combinada = mapa_2020.addBands(clima_seleccionado)

# Generar 1000 puntos aleatorios en nuestra Área de Interés (AOI)
puntos_aleatorios = ee.FeatureCollection.randomPoints(region=aoi, points=1000, seed=42)

# Extraer el valor exacto de la vegetación y el clima que hay debajo de cada uno de esos 1000 puntos
extraccion = imagen_combinada.sampleRegions(
    collection=puntos_aleatorios,
    scale=30, # MapBiomas usa una resolución de píxel de 30 metros
    geometries=True
)

# Mostrar un punto de muestra para ver si funcionó
print("Ejemplo de un punto extraído:")
print(extraccion.first().getInfo()['properties'])

Ejemplo de un punto extraído:
{'classification_2020': 19, 'temperature_2m': 295.7753985758358, 'total_precipitation_sum': 0.07040753416560008}



Finalmente, convertimos la colección de objetos espaciales (Features) de Earth Engine en un DataFrame de Pandas estructurado para poder entrenar nuestro modelo de Machine Learning localmente. Los datos se guardarán en un archivo CSV.

In [6]:
# Crear una carpeta local para guardar los datos si no existe
os.makedirs('data', exist_ok=True)
ruta_salida = 'data/muestra_chaco_2020.csv'

# 1. Traer los datos desde la nube a la memoria local de forma segura
info = extraccion.getInfo()
datos_crudos = info.get('features', [])

# 2. Extraer los valores iterando de forma explícita (para evitar errores de sintaxis)
lista_valores = []
for punto in datos_crudos:
    propiedades = punto.get('properties', {})
    lista_valores.append(propiedades)

# 3. Convertir a DataFrame de Pandas
df = pd.DataFrame(lista_valores)

# 4. Guardar como CSV
df.to_csv(ruta_salida, index=False)

print(f"¡Extracción exitosa! Se extrajeron {len(df)} puntos.")
print(f"Los datos se guardaron en: {ruta_salida}")

# Mostramos las primeras filas para verificar que todo funcionó
display(df.head())

¡Extracción exitosa! Se extrajeron 1000 puntos.
Los datos se guardaron en: data/muestra_chaco_2020.csv


,classification_2020,temperature_2m,total_precipitation_sum
0,19,295.775399,0.070408
1,3,295.586095,0.062030
2,3,296.062945,0.066857
3,3,295.879551,0.067924
4,19,296.440282,0.058736
